In [0]:
-- techniques that matter most in Databricks at scale.

-- slow version : scans entire table , no partitioning 

select * 
from raw.support_tickets 
where year(created_at) = 2024;

-- fast version: partition pruning + columnar selection 

select 
ticket_id , region , department , created_at, resolved_at
from raw.support_tickets
where created_at between '2024-01-01' and '2024-12-31'  -- partition filter 
and region is not null;

-- Use Zorder in databricks to co-locate frequently filtered columns 
optimize gold.tickets_clean
zorder by (region , created_at);
-- After this queries filtering on region + date are significantly faster 

-- Avoid re-scranning large tables: use a CTE to compute once , reuse 

with base as (
    select 
    region,
    date_trunc('month', created_at) as month,
    count(*) as total,
    sum(case when minutes_to_first_response <= 60 then 1 else 0 end) as within_sla
    from gold.tickets_clean
    group by 1, 2
),
with_pct as (
select 
region,
month,
total,
within_sla,
round(within_sla * 100.0 / total, 2)    as sla_pct
from base
)
select 
region,
total,
within_sla,
round(avg(sla_pct) over (
    partition by region
    order by month 
    rows between 2 preceding and current row ) , 2)    as rolling_3mo_sla
from with_pct
order by region, month;

-- Join tickets (from CRM) + agents ( from HR system) + regions (from ops table)
-- Real-world: each source may have different naming/formating 

select 
t.ticket_id,
t.region,
t.agent_id,
t.department,
t.minutes_to_first_response,
a.agent_name,
a.team,
r.region_head,
r.sla_target_minutes       -- SLA target varies by region 
from gold.tickets_clean       t     
left join silver.agents       a on t.agent_id = a.agent_id
left join ref.region_config   r on upper(trim(t.region)) == upper(trim(r.region_name))
-- upper/trim handles inconsistency between source systems 
where t.created_at >= date_sub(current_date, 90);

-- Handle ticket apearing in multiple source system (deduplication)

with deduped as (
    select *,
    row_number() over (
        partition by ticket_id
        order by updated_at desc   -- keep the most recent record 
    )     as rn 
    from (
        select ticket_id , status , updated_at, 'crm'     as source from raw.crm_tickets 
    union all 
    select ticket_id, status, updated_at , 'helpdesk'  as source from raw.helpdesk_tickets 
    )
)
select ticket_id , status, updated_at, source
from deduped
where rn =1 ;










